In [33]:
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

df=pd.DataFrame(fetch_california_housing().data,columns=fetch_california_housing().feature_names)
df['medianprice']=fetch_california_housing().target*100_000
x=df.iloc[:,:-1]
y=df.iloc[:,-1]
def train_test_validation(x,y,test_size=0.25,random_state=12):
    from sklearn.model_selection import train_test_split;x_,x_test,y_,y_test=train_test_split(x,y,test_size=test_size,random_state=random_state);x_train,x_val,y_train,y_val=train_test_split(x_,y_,test_size=(test_size/(1-test_size)),random_state=random_state);return x_train,x_test,x_val,y_train,y_test,y_val
x_train,x_test,x_val,y_train,y_test,y_val=train_test_validation(x,y)
df.head(3)

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,medianprice
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,452600.0
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,358500.0
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,352100.0


In [35]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

pipe1=make_pipeline(StandardScaler(),Lasso(random_state=12))
pipe1.fit(x_train,y_train)
ypred=pipe1.predict(x_val)
print(f"Before Tuning\nMeAbError: {mean_absolute_error(y_val,ypred)}\nMeSqError: {mean_squared_error(y_val,ypred)}\nR2 Score: {r2_score(y_val,ypred)}")

Before Tuning
MeAbError: 52999.01528329493
MeSqError: 5203219343.040534
R2 Score: 0.6054391495751872


In [ ]:
from sklearn.model_selection import GridSearchCV
params={
    'lasso__alpha':[0.00001,0.0001,0.001,0.01,0.1,1,5,10]
}
grid_search=GridSearchCV(estimator=pipe1,param_grid=params,scoring='neg_mean_absolute_error',n_jobs=-1,cv=5,verbose=0)
grid_search.fit(x_train,y_train)
ypred=grid_search.best_estimator_.predict(x_val)
print(f"After Tuning:\nMeAbError: {mean_absolute_error(y_val,ypred)}\nMeSqError: {mean_squared_error(y_val,ypred)}\nR2 Score: {r2_score(y_val,ypred)}\nBest params: {grid_search.best_params_}\nBest score: {grid_search.best_score_}")

After Tuning:
MeAbError: 52999.00782557901
MeSqError: 5203220085.118723
R2 Score: 0.6054390933032945
Best params: {'lasso__alpha': 1e-05}
Best score: -54618.13055086471
